# ACSAC 2026 Artifact Evaluation — *Brave New Browsing!*

**Paper:** "Brave New Browsing! Tracker Exposure under Browser-Agent Delegation" (ACSAC 2026)
**Repository:** https://github.com/0xk1h0/Browser_agent_measurement

This notebook is the zero-setup path for artifact evaluation. It runs the complete
Tier-1 reproduction on a free Google Colab CPU runtime: clone, install four pinned
Python packages, run `artifact/REPRODUCE.sh`, run `artifact/verify.py`, and print the reproduced tables.

**What it needs:** nothing. No GPU, no API keys, no mitmproxy, no browser automation,
no credentials. Network is used only to clone the repo and `pip install`; the analysis
itself is offline.

**How long it takes:** the reproduction step is a few seconds of compute
(~4 s on the authors' machine). The clone and `pip install` dominate wall clock.

**What it does:** regenerates every table and figure in the paper from the shipped
Tier-1 per-session aggregates, then diffs the regenerated files against the golden
copies committed in the repository. The analysis is deterministic: text/TSV/JSON
outputs come back byte-identical.

Run the cells top to bottom (Runtime -> Run all).

## 1. Clone the artifact and install the pinned dependencies

`artifact/requirements-repro.txt` pins the four packages used to produce the camera-ready
numbers (numpy, scipy, matplotlib, pandas). If Colab offers to restart the runtime
after the install, you can ignore it — every step below runs the analysis in a fresh
`python3` subprocess, so the notebook kernel's own imports do not matter.

In [ ]:
import pathlib, subprocess, sys

REPO_URL = "https://github.com/0xk1h0/Browser_agent_measurement.git"
BASE = pathlib.Path("/content") if pathlib.Path("/content").is_dir() else pathlib.Path.cwd()
REPO = BASE / "Browser_agent_measurement"
ART = REPO / "artifact" if (REPO / "artifact").is_dir() else REPO  # tolerate either layout

if (ART / "REPRODUCE.sh").exists():
    print("already cloned ->", REPO)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", str(ART / "requirements-repro.txt")], check=True)

print("\nrepo   :", REPO)
print("pinned :", (ART / "requirements-repro.txt").read_text().strip().splitlines()[-4:])

## 2. Run the reproduction

`artifact/REPRODUCE.sh` runs the four per-RQ drivers in sequence:

| Module | What it reproduces |
|--------|--------------------|
| `rq1_human_vs_agent` | Matched human-vs-agent pool: Table 1, Table 4, paired stats, CDF figure |
| `rq2_dose_response`  | 643-task WebVoyager dose-response: Table 2, Table 3, Figure 3, RTB and CMP summaries |
| `rq3_ablation`       | Action-space ablation: Table 6 + extended per-cell metrics, Table 12 |
| `cross_benchmark`    | Online Mind2Web validation: Table 7 |

**Expected last line:** `[REPRODUCE] All modules reproduced successfully.`

In [ ]:
import os

# REPRODUCE.sh defaults to `python3` from PATH; point it at the very interpreter
# we just installed the pinned packages into.
env = {**os.environ, "PYTHON": sys.executable}

r = subprocess.run(["bash", str(ART / "REPRODUCE.sh")], cwd=REPO, env=env, text=True,
                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(r.stdout)
print("exit code:", r.returncode)
assert r.returncode == 0, "REPRODUCE.sh failed - see the output above"

## 3. Verify against the golden outputs

`artifact/verify.py` diffs the freshly regenerated files against the copies committed under
each `*/expected_outputs/`. A pass means the numbers in the paper came out of this
code and this data, on this machine.

**Expected last line:** `[verify] All checks passed.`

In [ ]:
verify = ART / "verify.py"
if verify.exists():
    r = subprocess.run([sys.executable, str(verify)], cwd=ART, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(r.stdout)
    print("exit code:", r.returncode)
    assert r.returncode == 0, "verify.py reported a mismatch - see the output above"
else:
    print("verify.py is not present in this checkout - skipped.")
    print("You can still diff by hand:  git -C %s status --porcelain" % REPO)

## 4. The reproduced tables

Printed straight from the files the run just wrote. Compare them against the
corresponding tables in the paper.

In [ ]:
TABLES = [
    ("Table 1 - tracker hosts per task, matched 10-task pool (human vs 7 agents)",
     "rq1_human_vs_agent/expected_outputs/table1_trk_hosts.tsv"),
    ("Table 4 - tracker-family composition, matched pool",
     "rq1_human_vs_agent/expected_outputs/table4_family_composition.tsv"),
    ("Table 2 - per-agent exposure on 643 WebVoyager tasks",
     "rq2_dose_response/expected_outputs/table2.tsv"),
    ("Table 3 - action-space affordance bins",
     "rq2_dose_response/expected_outputs/table3.tsv"),
    ("Table 6 + extended metrics - per (agent, condition) ablation cell, 643-task scale",
     "rq3_ablation/data/rq3_643/full_metrics_per_cell.csv"),
    ("Table 12 (Appendix F) - schema vs proxy layer isolation",
     "rq3_ablation/data/rq3_layer_isolation/decomposition.csv"),
    ("Table 7 - Online Mind2Web cross-benchmark validation",
     "cross_benchmark/expected_outputs/table7.tsv"),
]

for title, rel in TABLES:
    path = ART / rel
    print("=" * 78)
    print(title)
    print(rel)
    print("=" * 78)
    print(path.read_text().rstrip() if path.exists() else "  (file not produced)")
    print()

## 5. Figures

The run also regenerates the paper figures (PDF for the paper, PNG for quick
viewing). Colab does not preview PDFs inline, so the cell below just lists what
was written; open or download any of them from the file browser on the left.

In [ ]:
figs = sorted(p for p in ART.glob("*/figures/*") if p.is_file())
for p in figs:
    print(f"{p.stat().st_size:>9,} B  {p.relative_to(ART)}")
print(f"\n{len(figs)} figure file(s).")

## What an evaluator should see

1. Cell 2 ends with `[REPRODUCE] All modules reproduced successfully.` (exit code 0).
2. Cell 3 ends with `[verify] All checks passed.` (exit code 0).
3. The tables printed in cell 4 match the corresponding tables in the paper.

### Scope

This notebook reproduces the **Tier-1** analysis: every table and figure in the paper
is derivable from the per-session aggregates shipped in the repository. It does *not*
re-run the measurement campaign itself (browser agents, mitmproxy capture, the JS
instrumentation shim), which needs GPUs, API keys, and live websites whose tracking
behavior changes daily — see `INSTALL.md` for that pipeline. Tier-3 raw flow bodies
and cookie values are not released under the study's IRB protocol.

### Other ways to run this

- **Locally:** `bash install.sh` then `bash artifact/REPRODUCE.sh`
- **Docker:** `docker build -t brave-new-browsing . && docker run --rm --network none brave-new-browsing`
  (the container needs the network only during `docker build`)

### If something goes wrong

- `pip install` warns about an incompatible preinstalled Colab package: harmless here.
  The analysis runs in a subprocess against the pinned versions just installed.
- A stale runtime from an earlier attempt: Runtime -> Disconnect and delete runtime,
  then run all cells again. Cell 1 re-uses an existing clone, so delete the
  `Browser_agent_measurement` folder if you want a genuinely clean checkout.
- Licensing: the code is MIT (`LICENSE`); the filter-list snapshots under
  `pipeline/classifier/tracker_lists/` keep their upstream licenses, recorded in
  `pipeline/classifier/tracker_lists/LICENSES.md`.